# Autoscaling diagnosis - "max workers, no speedup"

Interview question this notebook is a hands-on companion to:

> *"Your autoscaling cluster keeps hitting max workers, but jobs aren't faster.
> What's happening?"*

**Junior answer:** raise `max_workers`.

**Senior answer:** diagnose before you spend. Two mechanisms usually explain it:

1. Shuffle parallelism is capped by **partition count, not core count** - past that
   cap, extra workers just add idle cores.
2. **Skew**: stage wall time = the slowest task's time. A handful of straggler
   tasks pin the runtime while new workers sit idle on the bill.

Two sections below reproduce each mechanism live on the real `car_workshop` data,
then a third section connects them back to autoscaling with a live experiment you
run yourself. Keep the right panel open while a cell runs (not after):

- **Classic cluster:** Spark UI -> **Stages** (task count, Summary Metrics,
  Event Timeline) and **Executors** (idle cores).
- **Serverless:** **Query Profile** - same information, different UI.


In [0]:
import time

import pyspark.sql.functions as F
from pyspark.sql.window import Window

CATALOG = 'car_workshop'
LAB = f'{CATALOG}.lab'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {LAB}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {LAB}.files')
LAB_DIR = f'/Volumes/{CATALOG}/lab/files'


def timed(label, fn):
    t0 = time.time()
    result = fn()
    print(f'{label}: {time.time() - t0:.1f}s')
    return result


def n_partitions(df):
    """Count distinct physical partitions the rows actually landed in - no
    .rdd needed, so this works on serverless too (used in Section 1's
    SERVERLESS variant)."""
    return (df.withColumn('_pid', F.spark_partition_id())
              .select(F.countDistinct('_pid')).first()[0])


print(f'lab schema: {LAB}, lab volume: {LAB_DIR}')

dbutils.widgets.dropdown("Is cluster mode?", "False", ["True","False"])
is_cluster_mode = dbutils.widgets.get("Is cluster mode?")


## 1. Parallelism is capped by partition count, not core count

A shuffle stage's task count is fixed by `spark.sql.shuffle.partitions` (or AQE's
coalesced count) - full stop. Spark schedules exactly that many tasks for the
stage, no matter how many cores the cluster has. Extra workers only speed a stage
up while there is *unscheduled* work waiting for a free slot; once task count is
less than or equal to available cores, the remaining cores just sit idle - they
don't spawn more tasks out of nowhere.

This is the first half of the interview question: "bump the ceiling" assumes more
workers = more parallel tasks. That only holds if the stage actually has more
tasks than cores to run them on.


In [0]:
if is_cluster_mode == 'False':
    print("Skipping this section while it's not for serverless")
else:
    # =====================================================================
    # CLASSIC CLUSTER ONLY - FAILS on serverless:
    #   - spark.conf.set('spark.sql.shuffle.partitions', ...) is not on the
    #     serverless allowlist
    #   Serverless-friendly variant: NEXT CELL.
    # =====================================================================
    trx = spark.table(f'{CATALOG}.fact.fact_sales_transactions')
    total_cores = spark.sparkContext.defaultParallelism
    print(f'cluster total cores (defaultParallelism): {total_cores}')

    # LOW ceiling: force the shuffle stage down to 4 tasks, far below core count
    spark.conf.set('spark.sql.shuffle.partitions', 4)
    timed('groupBy with shuffle.partitions=4 (watch Stages: Tasks 4/4)',
          lambda: trx.groupBy('location_id').count().collect())

    # HIGH ceiling: same query, plenty of tasks to spread over every core
    spark.conf.set('spark.sql.shuffle.partitions', 400)
    timed('groupBy with shuffle.partitions=400 (watch Stages: Tasks up to 400)',
          lambda: trx.groupBy('location_id').count().collect())

    # NOTE: open Spark UI -> Stages WHILE each line above runs (not after) -
    # the "4/4" run leaves most cores idle in the Executors tab even though
    # total_cores is far higher than 4. Raising shuffle.partitions is what
    # fixes THIS problem - raising max_workers would not, because the extra
    # workers never get tasks to run.
    spark.conf.unset('spark.sql.shuffle.partitions')  # back to AQE-managed default


In [0]:
# SERVERLESS variant - spark.conf.set on shuffle.partitions is blocked and AQE
# is always on, so you cannot force a low ceiling manually. Instead: run the
# aggregation once and read the ACTUAL number of shuffle output partitions AQE
# picked, with n_partitions() (defined in the setup cell - no .rdd needed).
trx_df = spark.table(f'{CATALOG}.fact.fact_sales_transactions')
agg_df = trx_df.groupBy('location_id').count().cache()

timed('groupBy (AQE-managed shuffle.partitions)', lambda: agg_df.count())
print(f'AQE picked {n_partitions(agg_df)} output partitions for this shuffle')
print('cross-check the actual task count for the shuffle stage in Query Profile '
      '(serverless replacement for the classic Spark UI Stages view)')
agg_df.unpersist()


## 2. A straggler task pins the whole stage (skew)

Stage wall time = the **slowest** task's time, not the average. If one key
concentrates a disproportionate share of the rows, its task keeps grinding long
after every other task in the stage has finished - and those finished tasks' cores
sit idle waiting for the stage barrier. This is the second half of the interview
answer: "a handful of straggler tasks pin the runtime while the new workers sit
idle on the bill" - more workers cannot speed up a single task that is already
running alone.

`fact_invoices` has natural month buckets - the cell below multiplies one bucket's
rows to build a hot key on purpose, the same trick used in `data_skewness.ipynb`
(there, at multiplier=150, one task ended up with 643,243,725 rows - start smaller
here and dial it up once you've seen the effect).


In [0]:
dbutils.widgets.text(
    'SKEW_MULTIPLIER', '20',
    'Row multiplier for the hot bucket (data_skewness.ipynb used 150 -> one '
    'task with 643M rows - start small, raise it once you see the effect)')
SKEW_MULTIPLIER = int(dbutils.widgets.get('SKEW_MULTIPLIER'))

spark.sql(f"""
    CREATE OR REPLACE TABLE {LAB}.fact_invoices_skewed AS
    WITH cte AS (
      SELECT *,
             CASE WHEN month(sale_date) BETWEEN 1 AND 8  THEN 'skew_1'
                  WHEN month(sale_date) = 9                THEN 'skew_2'
                  WHEN month(sale_date) BETWEEN 10 AND 11  THEN 'skew_3'
                  ELSE 'skew_4'
             END AS skew_bucket
      FROM {CATALOG}.fact.fact_invoices
    ),
    hot AS (
      SELECT cte.*
      FROM cte
      CROSS JOIN LATERAL explode(sequence(1, {SKEW_MULTIPLIER})) AS t(multiplier)
      WHERE skew_bucket = 'skew_1'
    ),
    rest AS (
      SELECT * FROM cte WHERE skew_bucket != 'skew_1'
    )
    SELECT * FROM hot
    UNION ALL
    SELECT * FROM rest
""")

display(spark.table(f'{LAB}.fact_invoices_skewed')
        .groupBy('skew_bucket').count().orderBy(F.desc('count')))


In [0]:
inv = spark.table(f'{LAB}.fact_invoices_skewed')

# pathological: nearly all of skew_bucket='skew_1' must land in ONE task
w_skew = Window.partitionBy('skew_bucket').orderBy('sale_date')
timed(f'window over skew_bucket (SKEW_MULTIPLIER={SKEW_MULTIPLIER}, one hot task)',
      lambda: inv.withColumn('rn', F.row_number().over(w_skew))
                 .agg(F.max('rn')).collect())

# same work, healthy key: invoice_id has high cardinality -> spreads evenly
w_ok = Window.partitionBy('invoice_id').orderBy('sale_date')
timed('window over invoice_id (well distributed)',
      lambda: inv.withColumn('rn', F.row_number().over(w_ok))
                 .agg(F.max('rn')).collect())

# NOTE: while the first timed() call runs, open Spark UI -> Stages -> click into
# the stage -> Summary Metrics: max task duration will dwarf the median / 75th
# percentile. Then check the Event Timeline for one long bar while every other
# task finishes early and its executor sits idle waiting for the straggler.
# On serverless: the same signal lives in Query Profile's task-duration
# breakdown for this stage.


## 3. Why raising max workers doesn't fix either problem

Databricks autoscaling adds workers when there are more **pending** tasks than
free slots - that's the only signal it reacts to. Once the shuffle stage's task
count is capped (Section 1) or the stage is dominated by one or two stragglers
(Section 2), there's no pending-task backlog left to justify a new worker. The
cluster may still show a higher worker count from some other, bursty part of the
same job, but the bottleneck stage's wall time doesn't move. The extra workers
just add idle cores billed at the same rate - the "buys more idle cores" line
from the senior answer, made concrete.

**Go verify this on a real autoscaling cluster** (this step needs a cluster you
create - it can't be scripted safely from inside a notebook):

1. Create/attach a **classic** job cluster with autoscaling `min_workers=2`,
   `max_workers=8`.
2. Re-run Section 1's CLASSIC CLUSTER ONLY cell and Section 2's comparison cell
   on it.
3. Open the cluster's **Compute -> Metrics** tab and watch CPU utilization stay
   well under 100% even as the worker count climbs toward 8.
4. Check the cluster's **Event Log** for `SCALING` events firing while the
   bottleneck stage's wall time in Spark UI -> Stages stays flat.

That gap - workers climbing, utilization and wall time not - is the empirical
rebuttal to "just bump the ceiling."


## Wrap-up - what to check before you touch the ceiling

- **Job not faster at max workers** -> open Spark UI -> Stages and compare task
  count to core count *before* raising `max_workers` - a full ceiling is a
  symptom, not a diagnosis.
- **`shuffle.partitions` vs core count** -> partitions set the task ceiling for a
  shuffle stage; cores just decide how many of those tasks run at once. More
  cores than partitions means idle cores, not a faster stage.
- **Skew** -> Summary Metrics max/median ratio + one long bar in the Event
  Timeline, not "add workers." Fix is AQE skew join, broadcast, or salting (see
  `spark_deep_dive.ipynb` / `data_skewness.ipynb`) - not a bigger cluster.
- **Autoscaling earns its keep on bursty, variable load.** For steady-state jobs
  a right-sized fixed cluster is usually cheaper and more predictable than a
  wide min/max band chasing a bottleneck that scaling can't touch.


In [0]:
# cleanup - uncomment when you are done with this notebook
# spark.sql(f'DROP SCHEMA {LAB} CASCADE')   # drops lab tables AND the files volume
print(f'lab objects kept in {LAB} - drop the schema when done')
